# Ejemplo Practico — Colombianos Presos en el Exterior

## De la API al Analisis: Limpieza + Calidad + Visualizacion

Este notebook es un ejemplo completo resuelto. Tomamos datos reales de la API de datos.gov.co, los limpiamos, validamos su calidad y generamos un analisis. Cada paso esta explicado y ejecutado.

**Fuente:** Ministerio de Relaciones Exteriores — datos.gov.co

In [ ]:
!pip install pandera pydantic great_expectations

In [ ]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

---
## 1. Descargar datos desde la API

In [ ]:
# ============================================================
# CONSUMIR API CON PAGINACION
# ============================================================

# La API de Socrata retorna maximo N registros por llamada
# Usamos $limit y $offset para paginar hasta traer todo

url = "https://www.datos.gov.co/resource/e97j-vuf7.json"

registros = []
limit = 5000
offset = 0

while True:
    response = requests.get(url, params={"$limit": limit, "$offset": offset})
    response.raise_for_status()
    datos = response.json()
    if not datos:
        break
    registros.extend(datos)
    offset += limit

df = pd.DataFrame(registros)
print(f"Descargados: {len(df)} registros, {len(df.columns)} columnas")

---
## 2. Diagnostico inicial

In [ ]:
# ============================================================
# QUE TENEMOS
# ============================================================

df.info()

In [ ]:
df.head(3)

In [ ]:
# ============================================================
# PROBLEMAS DETECTADOS
# ============================================================

# 1. Nombres de columnas con encoding roto
print("Columnas originales:")
for col in df.columns:
    print(f"  {col}")

print(f"\n2. Todo es object (string), incluyendo cantidad, latitud, longitud")
print(f"3. Columna geocoded_column tiene dicts anidados")
print(f"4. 'DESCONOCIDO' aparece como valor en lugar de nulo")

# Contar DESCONOCIDO por columna
print("\nValores 'DESCONOCIDO' o similares por columna:")
for col in df.columns:
    n_desc = df[col].astype(str).str.contains('DESCONOCID', case=False, na=False).sum()
    if n_desc > 0:
        print(f"  {col}: {n_desc}")

In [ ]:
# ============================================================
# VALORES UNICOS POR COLUMNA CATEGORICA
# ============================================================

cols_categoricas = ['pais_prisi_n', 'consulado', 'delito',
                    'situaci_n_jur_dica', 'g_nero', 'grupo_edad',
                    'extraditado_y_o_repatriado']

for col in cols_categoricas:
    if col in df.columns:
        print(f"\n{col} ({df[col].nunique()} valores unicos):")
        print(df[col].value_counts().head(8).to_string())

---
## 3. Limpieza completa

In [ ]:
# ============================================================
# PASO 1: RENOMBRAR COLUMNAS
# ============================================================

# Los nombres originales tienen caracteres reemplazados por _
# Los cambiamos a nombres claros

mapeo_columnas = {
    'fecha_publicaci_n': 'fecha',
    'pais_prisi_n': 'pais',
    'consulado': 'consulado',
    'delito': 'delito',
    'extraditado_y_o_repatriado': 'extradicion',
    'situaci_n_jur_dica': 'situacion_juridica',
    'g_nero': 'genero',
    'grupo_edad': 'grupo_edad',
    'cantidad': 'cantidad',
    'latitud': 'latitud',
    'longitud': 'longitud',
}

df = df.rename(columns=mapeo_columnas)

# Eliminar columna anidada (ya tenemos lat/lon por separado)
if 'geocoded_column' in df.columns:
    df = df.drop(columns=['geocoded_column'])

print(f"Columnas renombradas: {df.columns.tolist()}")

In [ ]:
# ============================================================
# PASO 2: CONVERTIR TIPOS
# ============================================================

# La API retorna todo como string

# Fecha
df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')

# Cantidad a numerico
df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce')

# Coordenadas a float
df['latitud'] = pd.to_numeric(df['latitud'], errors='coerce')
df['longitud'] = pd.to_numeric(df['longitud'], errors='coerce')

print("Tipos despues de conversion:")
print(df[['fecha', 'cantidad', 'latitud', 'longitud']].dtypes)

In [ ]:
# ============================================================
# PASO 3: NULOS DISFRAZADOS
# ============================================================

# En este dataset los nulos no son NaN — son strings
# Hay que identificar cada variante y convertirla a NaN real

nulos_disfrazados = ['DESCONOCIDO', 'DESCONOCIDA', 'DESCONOCID']

# Reemplazar en columnas de texto
cols_texto = ['pais', 'consulado', 'delito', 'extradicion',
              'situacion_juridica', 'genero', 'grupo_edad']

for col in cols_texto:
    if col in df.columns:
        # Reemplazar valores que son exactamente DESCONOCIDO o similares
        df.loc[df[col].isin(nulos_disfrazados), col] = np.nan

# Coordenadas (0, 0) = punto en el Golfo de Guinea, no tiene sentido
# Esto pasa cuando el pais es desconocido
coords_cero = (df['latitud'] == 0) & (df['longitud'] == 0)
print(f"Registros con coordenadas (0, 0): {coords_cero.sum()}")
df.loc[coords_cero, ['latitud', 'longitud']] = np.nan

# Verificar nulos reales ahora
print("\nNulos reales despues de limpiar:")
nulos = df.isnull().sum()
print(nulos[nulos > 0].sort_values(ascending=False))

In [ ]:
# ============================================================
# PASO 4: NORMALIZAR TEXTO
# ============================================================

# 4a. Quitar prefijo "C. " y "BTA. " de consulados
df['consulado'] = (df['consulado']
    .str.replace(r'^C\.\s*', '', regex=True)
    .str.replace(r'^BTA\.\s*', 'BOGOTA ', regex=True)
    .str.strip()
)

print("Consulados despues de limpiar prefijo:")
print(df['consulado'].value_counts().head(10))

In [ ]:
# 4b. Normalizar delitos
# Hay variantes del mismo concepto por guion largo vs corto

print("Delitos con 'NO REPORTA':")
print(df[df['delito'].str.contains('NO REPORTA', na=False)]['delito'].unique())

# Unificar: reemplazar guion largo por guion corto
df['delito'] = df['delito'].str.replace('–', '-', regex=False)

# Verificar
print("\nDespues de unificar:")
print(df[df['delito'].str.contains('NO REPORTA', na=False)]['delito'].unique())

In [ ]:
# 4c. Verificar consistencia de las demas columnas categoricas

print("Genero:")
print(df['genero'].value_counts())

print("\nGrupo edad:")
print(df['grupo_edad'].value_counts())

print("\nExtradicion:")
print(df['extradicion'].value_counts())

print("\nSituacion juridica:")
print(df['situacion_juridica'].value_counts())

In [ ]:
# ============================================================
# PASO 5: DUPLICADOS
# ============================================================

n_dup = df.duplicated().sum()
print(f"Duplicados exactos: {n_dup}")

if n_dup > 0:
    df = df.drop_duplicates().reset_index(drop=True)
    print(f"Filas despues de deduplicar: {len(df)}")

In [ ]:
# ============================================================
# PASO 6: VERIFICACION FINAL
# ============================================================

print("=" * 50)
print("  DATASET LIMPIO")
print("=" * 50)
print(f"  Filas: {len(df)}")
print(f"  Columnas: {len(df.columns)}")
print(f"  Duplicados: {df.duplicated().sum()}")
print(f"  Nulos totales: {df.isnull().sum().sum()}")
print(f"\nTipos:")
print(df.dtypes)
print(f"\nValores unicos por columna:")
for col in df.columns:
    print(f"  {col}: {df[col].nunique()}")

df.info()

---
## 4. Validacion de calidad (Unidad 2)

In [ ]:
# ============================================================
# CONTRATO DE DATOS
# ============================================================

# Definimos las reglas que este dataset debe cumplir
# Si llegan datos nuevos, este contrato los valida

import pandera as pa

esquema_cancilleria = pa.DataFrameSchema({
    "fecha": pa.Column("datetime64[ns]", nullable=False),
    "pais": pa.Column(str, nullable=True),  # Puede ser nulo (antes era DESCONOCIDO)
    "consulado": pa.Column(str, nullable=False),
    "delito": pa.Column(str, nullable=True),
    "extradicion": pa.Column(str, nullable=True),
    "situacion_juridica": pa.Column(str, nullable=False),
    "genero": pa.Column(str, nullable=True),
    "grupo_edad": pa.Column(str, nullable=True),
    "cantidad": pa.Column(float, checks=pa.Check.greater_than(0), nullable=False),
    "latitud": pa.Column(float, checks=pa.Check.in_range(-90, 90), nullable=True),
    "longitud": pa.Column(float, checks=pa.Check.in_range(-180, 180), nullable=True),
})

try:
    esquema_cancilleria.validate(df, lazy=True)
    print("VALIDACION: APROBADA — el dataset cumple el contrato")
except pa.errors.SchemaErrors as e:
    print(f"VALIDACION: FALLO — {len(e.failure_cases)} errores")
    print(e.failure_cases.head(10))

In [ ]:
# ============================================================
# METRICAS DE CALIDAD
# ============================================================

# Resumen de calidad del dataset limpio

total_celdas = df.size
total_nulos = df.isnull().sum().sum()
completitud = (1 - total_nulos / total_celdas) * 100

print("Metricas de calidad:")
print(f"  Completitud: {completitud:.1f}%")
print(f"  Duplicados: {df.duplicated().sum()} ({df.duplicated().sum()/len(df)*100:.1f}%)")
print(f"  Filas con al menos un nulo: {df.isnull().any(axis=1).sum()} ({df.isnull().any(axis=1).sum()/len(df)*100:.1f}%)")

# Completitud por columna
print("\nCompletitud por columna:")
for col in df.columns:
    pct = (1 - df[col].isnull().sum() / len(df)) * 100
    barra = '#' * int(pct // 5) + '.' * (20 - int(pct // 5))
    print(f"  {col:25s} [{barra}] {pct:.0f}%")

---
## 5. Analisis y visualizacion

In [ ]:
# ============================================================
# TOP 10 PAISES — ponderado por cantidad
# ============================================================

# Cada fila tiene una columna 'cantidad' que indica cuantas personas
# representan ese registro. No es 1 fila = 1 persona.

top_paises = (df.groupby('pais')['cantidad']
              .sum()
              .sort_values(ascending=False)
              .head(10))

fig, ax = plt.subplots(figsize=(10, 5))
top_paises.plot(kind='barh', ax=ax, color='#007B3E')
ax.set_title('Top 10 Paises con Colombianos Presos', fontsize=14, fontweight='bold')
ax.set_xlabel('Cantidad de personas')
ax.invert_yaxis()

for i, (pais, val) in enumerate(top_paises.items()):
    ax.text(val + 10, i, f'{int(val):,}', va='center', fontsize=10)

ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TOP 10 DELITOS
# ============================================================

top_delitos = (df[df['delito'].notna()]
               .groupby('delito')['cantidad']
               .sum()
               .sort_values(ascending=False)
               .head(10))

fig, ax = plt.subplots(figsize=(10, 5))
top_delitos.plot(kind='barh', ax=ax, color='#DC2626')
ax.set_title('Top 10 Delitos', fontsize=14, fontweight='bold')
ax.set_xlabel('Cantidad de personas')
ax.invert_yaxis()

for i, (delito, val) in enumerate(top_delitos.items()):
    ax.text(val + 5, i, f'{int(val):,}', va='center', fontsize=10)

ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# DISTRIBUCION POR GENERO Y GRUPO DE EDAD
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Genero
genero = df[df['genero'].notna()].groupby('genero')['cantidad'].sum()
genero.plot(kind='bar', ax=axes[0], color=['#0EA5E9', '#D4A843'])
axes[0].set_title('Distribucion por Genero', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Cantidad de personas')
axes[0].tick_params(axis='x', rotation=0)
for i, val in enumerate(genero):
    axes[0].text(i, val + 20, f'{int(val):,}', ha='center', fontweight='bold')

# Grupo de edad
orden_edad = ['PRIMERA INFANCIA', 'INFANTE', 'ADOLESCENTE',
              'ADULTO JOVEN', 'ADULTO', 'ADULTO MAYOR']
edad = (df[df['grupo_edad'].notna()]
        .groupby('grupo_edad')['cantidad']
        .sum()
        .reindex(orden_edad)
        .dropna())
edad.plot(kind='bar', ax=axes[1], color='#007B3E')
axes[1].set_title('Distribucion por Grupo de Edad', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Cantidad de personas')
axes[1].tick_params(axis='x', rotation=45)
for i, val in enumerate(edad):
    axes[1].text(i, val + 10, f'{int(val):,}', ha='center', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SITUACION JURIDICA
# ============================================================

sit = (df.groupby('situacion_juridica')['cantidad']
       .sum()
       .sort_values(ascending=True))

fig, ax = plt.subplots(figsize=(10, 4))
colores_sit = ['#007B3E' if 'CONDENADO' in s else
               '#D4A843' if 'JUICIO' in s else
               '#0EA5E9' if 'INVESTIGACIÓN' in s else '#999'
               for s in sit.index]
sit.plot(kind='barh', ax=ax, color=colores_sit)
ax.set_title('Situacion Juridica', fontsize=14, fontweight='bold')
ax.set_xlabel('Cantidad de personas')
ax.spines[['top', 'right']].set_visible(False)

for i, val in enumerate(sit):
    ax.text(val + 5, i, f'{int(val):,}', va='center', fontsize=10)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# TABLA CRUZADA: DELITO x PAIS (TOP 5)
# ============================================================

# Para los 5 paises con mas colombianos, cual es el delito principal?

top5_paises = top_paises.head(5).index.tolist()
top5_delitos = top_delitos.head(5).index.tolist()

# Filtrar
df_cruce = df[
    (df['pais'].isin(top5_paises)) &
    (df['delito'].isin(top5_delitos))
]

# Tabla cruzada ponderada por cantidad
cruce = pd.pivot_table(
    df_cruce,
    values='cantidad',
    index='pais',
    columns='delito',
    aggfunc='sum',
    fill_value=0
)

# Heatmap
fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(cruce, annot=True, fmt='.0f', cmap='YlGn',
            linewidths=0.5, ax=ax)
ax.set_title('Cantidad por Pais y Delito (Top 5)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# EXTRADICIONES
# ============================================================

extradiciones = df[df['extradicion'] == 'EXTRADICION']

total_extradiciones = extradiciones['cantidad'].sum()
print(f"Total de extradiciones: {int(total_extradiciones)}")

print("\nExtradiciones por pais:")
ext_pais = extradiciones.groupby('pais')['cantidad'].sum().sort_values(ascending=False)
print(ext_pais.to_string())

print("\nExtradiciones por delito:")
ext_delito = extradiciones.groupby('delito')['cantidad'].sum().sort_values(ascending=False)
print(ext_delito.to_string())

In [ ]:
# ============================================================
# MAPA — scatter con coordenadas
# ============================================================

# Agrupar por pais con coordenadas validas
df_mapa = (df[df['latitud'].notna()]
           .groupby(['pais', 'latitud', 'longitud'])
           .agg(total=('cantidad', 'sum'))
           .reset_index())

fig, ax = plt.subplots(figsize=(14, 7))

scatter = ax.scatter(
    df_mapa['longitud'],
    df_mapa['latitud'],
    s=df_mapa['total'] * 0.5,  # Tamano proporcional a cantidad
    c=df_mapa['total'],
    cmap='YlOrRd',
    alpha=0.6,
    edgecolors='gray',
    linewidth=0.5
)

# Etiquetar los top 5
for _, row in df_mapa.nlargest(5, 'total').iterrows():
    ax.annotate(f"{row['pais']}\n({int(row['total'])})",
                xy=(row['longitud'], row['latitud']),
                textcoords='offset points', xytext=(10, 10),
                fontsize=8, fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='gray', lw=0.5))

ax.set_title('Colombianos Presos en el Exterior — Distribucion Geografica',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Longitud')
ax.set_ylabel('Latitud')
plt.colorbar(scatter, label='Cantidad de personas')
plt.tight_layout()
plt.show()

---
## 6. Pipeline completo encapsulado

In [ ]:
# ============================================================
# TODO EN UNA FUNCION — reutilizable
# ============================================================

def pipeline_cancilleria(url="https://www.datos.gov.co/resource/e97j-vuf7.json"):
    """
    Pipeline completo: descarga, limpia y retorna los datos
    de colombianos presos en el exterior.
    """
    # 1. Descargar
    registros = []
    offset = 0
    while True:
        r = requests.get(url, params={"$limit": 5000, "$offset": offset})
        r.raise_for_status()
        datos = r.json()
        if not datos:
            break
        registros.extend(datos)
        offset += 5000
    
    df = pd.DataFrame(registros)
    
    # 2. Renombrar
    df = df.rename(columns={
        'fecha_publicaci_n': 'fecha', 'pais_prisi_n': 'pais',
        'situaci_n_jur_dica': 'situacion_juridica',
        'g_nero': 'genero', 'extraditado_y_o_repatriado': 'extradicion',
    })
    if 'geocoded_column' in df.columns:
        df = df.drop(columns=['geocoded_column'])
    
    # 3. Tipos
    df['fecha'] = pd.to_datetime(df['fecha'], errors='coerce')
    df['cantidad'] = pd.to_numeric(df['cantidad'], errors='coerce')
    df['latitud'] = pd.to_numeric(df['latitud'], errors='coerce')
    df['longitud'] = pd.to_numeric(df['longitud'], errors='coerce')
    
    # 4. Nulos disfrazados
    for col in ['pais', 'consulado', 'delito', 'extradicion',
                'situacion_juridica', 'genero', 'grupo_edad']:
        if col in df.columns:
            df.loc[df[col].isin(['DESCONOCIDO', 'DESCONOCIDA']), col] = np.nan
    
    df.loc[(df['latitud'] == 0) & (df['longitud'] == 0), ['latitud', 'longitud']] = np.nan
    
    # 5. Normalizar texto
    df['consulado'] = (df['consulado']
        .str.replace(r'^C\.\s*', '', regex=True)
        .str.replace(r'^BTA\.\s*', 'BOGOTA ', regex=True)
        .str.strip())
    df['delito'] = df['delito'].str.replace('–', '-', regex=False)
    
    # 6. Duplicados
    df = df.drop_duplicates().reset_index(drop=True)
    
    return df

# Ejecutar
df_limpio = pipeline_cancilleria()
print(f"Pipeline ejecutado: {len(df_limpio)} filas, {len(df_limpio.columns)} columnas")
print(f"Nulos: {df_limpio.isnull().sum().sum()}")
print(f"Duplicados: {df_limpio.duplicated().sum()}")

In [ ]:
# Guardar
df_limpio.to_csv('colombianos_exterior_limpio.csv', index=False)
df_limpio.to_parquet('colombianos_exterior_limpio.parquet')
print("Guardado en CSV y Parquet")

---
## Resumen de lo aplicado

| Tema del curso | Que aplicamos |
|---|---|
| **1.1 Python para datos** | requests para consumir API, pandas para cargar y explorar, funciones para encapsular |
| **1.2 Limpieza** | Renombrar columnas, convertir tipos, nulos disfrazados (DESCONOCIDO, coords 0,0), normalizar texto (guiones, prefijos), deduplicar |
| **1.3 Visualizacion** | Barras horizontales, scatter con tamano, heatmap cruzado, distribucion por genero y edad |
| **2.1 Calidad** | Contrato con pandera, validacion automatica, metricas de completitud |
| **2.2 Diseno** | Pipeline encapsulado en funcion reutilizable, export a Parquet |